# 6. Pace & outcomes: shots

Compare shot 5v5 attempts by total pace in the preceding five seconds, mean shot distance, and on-net percentage.

## 6.1. Load inputs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROCESSED_DATA_DIR = Path("../data/processed")
SHOT_LOOKBACK_SECONDS = 5

# Event IDs preserve order, including events sharing a clock value.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
five_v_five_shots = events.loc[
    events["event"].isin(["Shot", "Goal"]) & events["is_5v5"]
].copy()
excluded_deflections = int(five_v_five_shots["detail_1"].eq("Deflection").sum())
shots = five_v_five_shots.loc[~five_v_five_shots["detail_1"].eq("Deflection")].copy()

## 6.2. Calculate preceding total pace

Pool same-possession movements ending within the previous five secs.

In [2]:
def add_preceding_pace(shot_events, transition_events):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for shot in shot_events.itertuples(index=False):
        # Countdown clocks run backward; event IDs exclude later same-clock events.
        history = movement_groups.get(shot.possession_id, transition_events.iloc[:0])
        history = history.loc[
            history["end_event_id"].le(shot.event_id)
            & history["end_modeled_clock_seconds"].between(
                shot.modeled_clock_seconds,
                shot.modeled_clock_seconds + SHOT_LOOKBACK_SECONDS,
            )
        ]
        distance = float(history["distance_ft"].sum())
        duration = float(history["modeled_elapsed_seconds"].sum())
        rows.append(
            {
                "event_id": shot.event_id,
                "preceding_transitions": len(history),
                "preceding_distance_ft": distance,
                "preceding_movement_seconds": duration,
                "preceding_total_pace_ft_s": distance / duration
                if duration > 0
                else np.nan,
            }
        )
    return shot_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


shots = add_preceding_pace(shots, movements)

## 6.3. Identify shot outcomes and distance

Goals and On Net shots are successes. Blocked and missed attempts are misses.

Measure shot distance to the attacking net center `(189, 42.5)`

In [3]:
shots["shot_outcome"] = shots["detail_2"].replace({"On Net": "On net"})
shots.loc[shots["event"].eq("Goal"), "shot_outcome"] = "On net"
assert shots["shot_outcome"].isin(["Blocked", "Missed", "On net"]).all()
shots["on_net"] = shots["shot_outcome"].eq("On net")
valid_location = (shots["x"].between(0, 200) & shots["y"].between(0, 85)).fillna(False)
shots["distance_to_net_ft"] = np.hypot(shots["x"] - 189, shots["y"] - 42.5).where(
    valid_location
)

## 6.4. Summarize quartiles

In [4]:
def summarize_shots(data):
    eligible = data.loc[data["preceding_total_pace_ft_s"].notna()].copy()
    bands, edges = pd.qcut(
        eligible["preceding_total_pace_ft_s"],
        q=4,
        labels=False,
        retbins=True,
        duplicates="drop",
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    summary = (
        eligible.loc[eligible["quartile"].notna()]
        .groupby("quartile", observed=True)
        .agg(
            attempts=("event_id", "size"),
            on_net_attempts=("on_net", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=("preceding_total_pace_ft_s", "median"),
            mean_distance_ft=("distance_to_net_ft", "mean"),
            distance_attempts=("distance_to_net_ft", "count"),
        )
        .reset_index()
    )
    summary["on_net_pct"] = 100 * summary["on_net_attempts"] / summary["attempts"]
    return summary.merge(boundaries, on="quartile", validate="one_to_one"), boundaries


shot_summary, quartile_boundaries = summarize_shots(shots)
# Show exclusions separately from genuine zero-pace attempts.
coverage_df = pd.DataFrame(
    [
        {
            "five_v_five_attempts": len(five_v_five_shots),
            "excluded_deflections": excluded_deflections,
            "non_deflected_attempts": len(shots),
            "missing_pace": int(shots["preceding_total_pace_ft_s"].isna().sum()),
            "zero_pace": int(shots["preceding_total_pace_ft_s"].eq(0).sum()),
            "charted": int(shot_summary["attempts"].sum()),
        }
    ]
)
display(coverage_df)

,five_v_five_attempts,excluded_deflections,non_deflected_attempts,missing_pace,zero_pace,charted
0,3144,117,3027,107,14,2920


## 6.5. Display shot outcomes

Both panels use the same attempts and quartiles, Q1 slowest to Q4 fastest. Hover shows counts and pace boundaries. Empty groups remain gaps; fewer available bands are disclosed.

In [5]:
QUARTILE_COLORS = ["#bfdbfe", "#93c5fd", "#3b82f6", "#1d4ed8"]


def shot_outcomes_figure(summary, boundaries):
    panels = [
        ("mean_distance_ft", "Mean shot distance (ft)"),
        ("on_net_pct", "On-net attempts (%)"),
    ]
    figure = make_subplots(rows=1, cols=2, subplot_titles=[p[1] for p in panels])
    bars = summary.set_index("quartile").reindex(range(1, len(boundaries) + 1))
    customdata = bars[
        [
            "attempts",
            "on_net_attempts",
            "games",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
            "median_pace_ft_s",
            "distance_attempts",
        ]
    ].to_numpy()
    for column, (metric, label) in enumerate(panels, start=1):
        suffix = "%" if metric == "on_net_pct" else ""
        figure.add_trace(
            go.Bar(
                x=[f"Q{q}" for q in bars.index],
                y=bars[metric],
                marker_color=QUARTILE_COLORS[: len(boundaries)],
                showlegend=False,
                text=[
                    f"{value:.1f}{suffix}" if pd.notna(value) else ""
                    for value in bars[metric]
                ],
                textposition="outside",
                cliponaxis=False,
                customdata=customdata,
                hovertemplate=(
                    f"%{{x}}<br>{label}: %{{y:.1f}}{suffix}"
                    "<br>Attempts: %{customdata[0]:.0f}"
                    "<br>On-net attempts: %{customdata[1]:.0f}"
                    "<br>Games: %{customdata[2]:.0f}"
                    "<br>Pace range: %{customdata[3]:.2f} to %{customdata[4]:.2f} ft/s"
                    "<br>Median pace: %{customdata[5]:.1f} ft/s"
                    "<br>Attempts with distance: %{customdata[6]:.0f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
        figure.update_yaxes(title_text=label, rangemode="tozero", row=1, col=column)
    figure.update_yaxes(range=[0, 100], row=1, col=2)
    figure.update_xaxes(
        title_text="Preceding total pace quartile",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    if summary.empty:
        note = "No data: no measurable pace with assigned bands."
    elif len(boundaries) < 4:
        note = f"Only {len(boundaries)} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title="Pace preceding shots | previous 5 modeled seconds | 5v5, non-deflected",
        template="plotly_white",
        height=500,
        margin=dict(t=100, b=100),
    )
    return figure


shot_figure = shot_outcomes_figure(shot_summary, quartile_boundaries)
shot_figure.show()

## 6.6. Export shot outcomes

In [6]:
shot_outcomes_df = (
    shot_summary[
        [
            "quartile",
            "attempts",
            "on_net_attempts",
            "games",
            "median_pace_ft_s",
            "mean_distance_ft",
            "distance_attempts",
            "on_net_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "quartile": "Int64",
            "attempts": "Int64",
            "on_net_attempts": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "mean_distance_ft": "Float64",
            "distance_attempts": "Int64",
            "on_net_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values("quartile")
    .reset_index(drop=True)
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_path = PROCESSED_DATA_DIR / "shot_outcomes.parquet"
shot_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, shot_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 4 rows and 10 columns to ..\data\processed\shot_outcomes.parquet


,quartile,attempts,on_net_attempts,games,median_pace_ft_s,mean_distance_ft,distance_attempts,on_net_pct,lower_pace_ft_s,upper_pace_ft_s
0,1,730,322,34,9.507363,36.488357,730,44.109589,0.0,13.459216
1,2,730,336,34,16.198034,36.108566,730,46.027397,13.459216,18.340289
2,3,730,345,34,20.598752,35.7686,730,47.260274,18.340289,22.951698
3,4,730,368,34,26.384149,35.775928,730,50.410959,22.951698,172.372934
